In [ ]:
# ============================================================
#  Project Name: brandalyze
#  File: tweet_model_training.ipynb
#  Author: dngi (https://twitter.com/_dngi)
#  Created: <2026-02-13>
#  
#  Copyright (c) 2026 Dom G. (https://twitter.com/_dngi)
#  All rights reserved.
#
#  This notebook and its contents are confidential and
#  proprietary. Unauthorized copying, distribution,
#  modification, or use is strictly prohibited.
# ============================================================

# Brandalyze Tweet Model Training

## Setup and Data Loading
- loading the labeled dataset and prepping for ML training

In [ ]:
import json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# load data
with open('labeled-viral-tweets.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

df = pd.DataFrame(data)

# filter only labeled tweets
df = df[df['labels'].notna()].copy()

## Prepare Training Data
- Extract features and labels for each classification task
- clean data by removing any `unknown` labels

In [ ]:
# extract labels
df['format'] = df['labels'].apply(lambda x: x.get('format'))
df['hookQuality'] = df['labels'].apply(lambda x: x.get('hookQuality'))
df['closerType'] = df['labels'].apply(lambda x: x.get('closerType'))

# prepare text input
df['text_clean'] = df['text'].str.strip()

df = df[df['format'] != 'unknown']
df = df[df['hookQuality'] != 'unknown']
df = df[df['closerType'] != 'unknown']

print(f"total training samples: {len(df)}")

## Train/Val/Test Split
- Split data using stratification to maintain class balance

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    random_state=42,
    stratify=df['format']
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df['format']
)

print(f"Train: {len(train_df)} ({len(train_df) / len(df)*100:.1f}%)")
print(f"Val: {len(val_df)} ({len(val_df) / len(df)*100:.1f}%)")
print(f"Test: {len(test_df)} ({len(test_df) / len(df)*100:.1f}%)")

## Model 1: Format Classifier

In [ ]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# tokenize
train_encodings = tokenizer(
    train_df['text_clean'].tolist(),
    truncation=True,
    padding=True,
    max_length=280
)

val_encodings = tokenizer(
    val_df['text_clean'].tolist(),
    truncation=True,
    padding=True,
    max_length=280
)

test_encodings = tokenizer(
    test_df['text_clean'].tolist(),
    truncation=True,
    padding=True,
    max_length=280
)

label_encoder = LabelEncoder()
train_labels = label_encoder.fit_transform(train_df['format'])
val_labels = label_encoder.transform(val_df['format'])
test_labels = label_encoder.transform(test_df['format'])

# create a pytorch dataset
import torch

class TweetDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    
    def __len__(self):
        return len(self.labels)
    
train_dataset = TweetDataset(train_encodings, train_labels)
val_dataset = TweetDataset(val_encodings, val_labels)
test_dataset = TweetDataset(test_encodings, test_labels)


## Training Config
- setting up training params

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='./models/format_classifier',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_encoder.classes_)
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

## Train and Evaluate

- run training and check metrics

In [ ]:
trainer.train()

# evaluate on the test set
predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)

from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(
    test_labels,
    pred_labels,
    target_names=label_encoder.classes_
))

# save model
model.save_pretrained('./models/format_classifier_final')
tokenizer.save_pretrained('./models/format_classifier_final')

# save label encoder
import joblib
joblib.dump(label_encoder, './models/format_label_encoder.pkl')

## Model 2: Hook Quality Classifier

In [ ]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# tokenize
train_encodings = tokenizer(
    train_df['text_clean'].tolist(),
    truncation=True,
    padding=True,
    maxLength=280
)

val_encodings = tokenizer(
    val_df['text_clean'].tolist(),
    truncation=True,
    padding=True,
    maxLength=280
)


test_encodings = tokenizer(
    test_df['text_clean'].tolist(),
    truncation=True,
    padding=True,
    maxLength=280
)

label_encoder = LabelEncoder()
train_labels = label_encoder.fit_transform(train_df['hookQuality'])
val_labels = label_encoder.fit_transform(val_df['hookQuality'])
test_labels = label_encoder.fit_transform(test_df['hookQuality'])

# create a pytorch dataset
import torch

class TweetDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    
    def __len__(self):
        return len(self.labels)
    
train_dataset = TweetDataset(train_encodings, train_labels)
val_dataset = TweetDataset(val_encodings, val_labels)
test_dataset = TweetDataset(test_encodings, test_labels)

## Training Config
- hookQuality classifier training params

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='./models/hook_classifier',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_encoder.classes_)
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

## Train and Evaluate
- run training on hookQuality classifier and check metrics

In [ ]:
trainer.train()

# evaluate on the test set
predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)

from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(
    test_labels,
    pred_labels,
    target_names=label_encoder.classes_
))

# save model
model.save_pretrained('./models/hook_classifier_final')
tokenizer.save_pretrained('./models/hook_classifier_final')

import joblib
joblib.dump(label_encoder, './models/hook_label_encoder.pkl')

## Model 3: Closer Type Classifier

In [ ]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# tokenize
train_encodings = tokenizer(
    train_df['text_clean'].tolist(),
    truncation=True,
    padding=True,
    maxLength=280
)

val_encodings = tokenizer(
    val_df['text_clean'].tolist(),
    truncation=True,
    padding=True,
    maxLength=280
)

test_encodings = tokenizer(
    test_df['text_clean'].tolist(),
    truncation=True,
    padding=True,
    maxLength=280
)

label_encoder = LabelEncoder()
train_labels = label_encoder.fit_transform(train_df['closerType'])
val_labels = label_encoder.fit_transform(val_df['closerType'])
test_labels = label_encoder.fit_transform(test_df['closerType'])

# create a pytorch
import torch

class TweetDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = val_labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    
    def __len__(self):
        return len(self.labels)
    
train_dataset = TweetDataset(train_encodings, train_labels)
val_dataset = TweetDataset(val_encodings, val_labels)
test_dataset = TweetDataset(test_encodings, test_labels)

## Training Config
- closerType classifier training params

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='./models/closer_classifier',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_encoder.classes_)
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

## Train and Evaluate
- run training on the closerType classifier and check metrics

In [ ]:
trainer.train()

# evaluate on the test set
predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)

from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(
    test_labels,
    pred_labels,
    target_names=label_encoder.classes_
))

# save model
model.save_pretrained('./models/closer_classifier_final')
tokenizer.save_pretrained('./models/closer_classifier_final')

import joblib
joblib.dump(label_encoder, './models/closer_label_encoder.pkl')